# JAO Feature Engineering

JAO publishes the day-ahead forecast at 10:30 on D-1, so every series is already known for delivery day D. Features stay on hour t. A 24-hour change is the value at t minus the value 24 hours earlier. `create_all_jao_features` builds every series once.

The TFT feature set grows in three steps. D1 is the Polish market and macro factors, with no JAO column. D2 adds Polish market coupling. D3 adds foreign markets and systems. JAO supplies D2 and part of D3, so two files are saved: `d2` and `d3`. Each file holds only the columns its step adds. `select_features(df, model, step)` keeps one of them: `model` is `"tft"` and `step` is `"d2"` or `"d3"`.

**Inputs**

* One merged JAO frame: core net positions, the Polish min/max net-position limits, reference-program border flows, D2CF generation and vertical load, and the Polish allocation constraints.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent.parent))

from src.config import (
    DATA_DIR,
    DOWNLOAD_START_DATE,
    DOWNLOAD_END_DATE,
)
from src.utils.file_io import (
    load_merged_data,
    save_merged_data,
    inspect_merged_file,
)
from src.utils.feature_engineering.fe_jao import (
    COLUMNS,
    create_all_jao_features,
    select_features,
)

In [2]:
date_range = f"{DOWNLOAD_START_DATE}_{DOWNLOAD_END_DATE}"

data_jao = load_merged_data(
    DATA_DIR / "preprocessed" / "jao" / date_range / "jao.parquet"
)

## 1. Input Columns

The merged JAO frame, before any feature is derived from it.

In [3]:
print(list(data_jao.columns))

['allocation_constraint_PL_import', 'allocation_constraint_PL_export', 'd2cf_verticalLoad_AT', 'd2cf_verticalLoad_BE', 'd2cf_verticalLoad_CZ', 'd2cf_verticalLoad_DE', 'd2cf_verticalLoad_FR', 'd2cf_verticalLoad_HR', 'd2cf_verticalLoad_HU', 'd2cf_verticalLoad_NL', 'd2cf_verticalLoad_PL', 'd2cf_verticalLoad_RO', 'd2cf_verticalLoad_SI', 'd2cf_verticalLoad_SK', 'd2cf_generation_AT', 'd2cf_generation_BE', 'd2cf_generation_CZ', 'd2cf_generation_DE', 'd2cf_generation_FR', 'd2cf_generation_HR', 'd2cf_generation_HU', 'd2cf_generation_NL', 'd2cf_generation_PL', 'd2cf_generation_RO', 'd2cf_generation_SI', 'd2cf_generation_SK', 'd2cf_coreNetPosition_AT', 'd2cf_coreNetPosition_BE', 'd2cf_coreNetPosition_CZ', 'd2cf_coreNetPosition_DE', 'd2cf_coreNetPosition_FR', 'd2cf_coreNetPosition_HR', 'd2cf_coreNetPosition_HU', 'd2cf_coreNetPosition_NL', 'd2cf_coreNetPosition_PL', 'd2cf_coreNetPosition_RO', 'd2cf_coreNetPosition_SI', 'd2cf_coreNetPosition_SK', 'd2cf_verticalLoad_50Hertz', 'd2cf_verticalLoad_Ampri

## 2. Features

**Poland and Polish borders.** Core net position `np_PL`, day-ahead generation `gen_PL`, vertical load `vload_PL`, the import and export limits `min_PL` and `max_PL`, and the allocation constraints `ac_PL_import` and `ac_PL_export`. Border flows are signed so that positive means flow into Poland: `refprog_border_DE_PL` as published, and the reverse of `refprog_border_PL_CZ`, `refprog_border_PL_SK` and `refprog_border_PL_UA`, stored as `flow_DE_PL`, `flow_CZ_PL`, `flow_SK_PL` and `flow_UA_PL`.

**Foreign.** Core net positions of Germany (`np_DE`), 50Hertz (`np_50Hz`), Czechia (`np_CZ`) and Slovakia (`np_SK`). Flows between foreign zones, positive in the direction of the name: `flow_DE_CZ` is `refprog_border_DE_CZ` and `flow_CZ_SK` is `refprog_border_CZ_SK`.

In [4]:
features = create_all_jao_features(data_jao)

In [5]:
print(features.shape)
print(list(features.columns))

(38304, 17)
['np_PL', 'np_DE', 'np_CZ', 'np_SK', 'np_50Hz', 'min_PL', 'max_PL', 'gen_PL', 'vload_PL', 'flow_DE_PL', 'flow_CZ_PL', 'flow_SK_PL', 'flow_UA_PL', 'flow_DE_CZ', 'flow_CZ_SK', 'ac_PL_import', 'ac_PL_export']


## 3. The TFT Steps

`COLUMNS["tft"]` lists the JAO part of each step.

**D2 (11 columns).** Polish market coupling: `np_PL`, `gen_PL`, `vload_PL`, `ac_PL_import`, `ac_PL_export`, `flow_DE_PL`, `flow_CZ_PL`, `flow_SK_PL`, `flow_UA_PL`, `min_PL`, `max_PL`.

**D3 (6 columns).** Foreign systems: `np_DE`, `np_50Hz`, `np_CZ`, `np_SK`, `flow_DE_CZ`, `flow_CZ_SK`.

In [6]:
for model, steps in COLUMNS.items():
    for step in steps:
        subset = select_features(features, model, step)
        print(f"=== {model}/{step} ===")
        print(subset.shape[1], "features")
        print(list(subset.columns))

=== tft/d2 ===
11 features
['np_PL', 'gen_PL', 'vload_PL', 'ac_PL_import', 'ac_PL_export', 'flow_DE_PL', 'flow_CZ_PL', 'flow_SK_PL', 'flow_UA_PL', 'min_PL', 'max_PL']
=== tft/d3 ===
6 features
['np_DE', 'np_50Hz', 'np_CZ', 'np_SK', 'flow_DE_CZ', 'flow_CZ_SK']


## 4. Conclusions

D2 and D3 are saved for TFT, two files in total. Files go to `data/features/jao/tft/{d2|d3}/`, and each file is verified by reading it back.

## 5. Saving Feature Data

In [7]:
for model, steps in COLUMNS.items():
    for step in steps:
        path = DATA_DIR / "features" / "jao" / model / step / f"{date_range}.parquet"
        save_merged_data(select_features(features, model, step), path)
        inspect_merged_file(path)

  [FILE] data/features/jao/tft/d2/2022-06-09_2026-10-21.parquet | shape: (38304, 11) | idx: 2022-06-08 22:00:00+00:00-2026-10-21 21:00:00+00:00 | cols: 11 | NaN: 216
  [FILE] data/features/jao/tft/d3/2022-06-09_2026-10-21.parquet | shape: (38304, 6) | idx: 2022-06-08 22:00:00+00:00-2026-10-21 21:00:00+00:00 | cols: 6 | NaN: 144
